<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/Continual_Shared_LoRA_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Continual Learning Demo: Standard LoRA vs Shared Subspace LoRA

This notebook demonstrates:
- Task-specific classification heads (no overwriting)
- Standard LoRA sequential fine-tuning
- Shared LoRA with stability regularization (L2 constraint)
- Automatic catastrophic forgetting table

Datasets used: GLUE CoLA → MRPC → SST-2
Backbone: roberta-base


To fix the `Invalid Notebook` error when saving to GitHub, we will remove the `metadata.widgets` entry from the notebook's `.ipynb` file. This is a common issue that arises when widget states are not fully compatible with GitHub's rendering engine or `nbconvert`'s validation rules.

**Note:** This action will clear any saved states of interactive widgets (e.g., sliders, progress bars) from the notebook's metadata.

import json
import os

# Get the path to the current notebook
notebook_path = os.getenv('COLAB_JUPYTER_SERVER_ROOT') + os.getenv('COLAB_NOTEBOOK_PATH')

# Read the notebook content
with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook_content = json.load(f)

# Check if 'metadata.widgets' exists and remove it
if 'widgets' in notebook_content.get('metadata', {}):
    del notebook_content['metadata']['widgets']
    print("Removed 'metadata.widgets' from the notebook.")
else:
    print("'metadata.widgets' not found or already removed.")

# Save the modified notebook content back to the file
with open(notebook_path, 'w', encoding='utf-8') as f:
    json.dump(notebook_content, f, indent=4)

print("Notebook metadata updated. Please try saving to GitHub again.")

After running the above cell, please try saving your notebook to GitHub again. If the issue persists, you might need to restart your Colab runtime and then save, or check for any other validation errors that might arise.

In [1]:
!pip -q install transformers datasets peft evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model

if torch.cuda.is_available():
    device = "cuda"
    num_gpus = torch.cuda.device_count()
    print(f"Using {num_gpus} GPU(s).")
else:
    device = "cpu"
    print("Using CPU.")

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Using CPU.


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
tasks = [
    ("cola", "glue", "cola", 2),
    ("mrpc", "glue", "mrpc", 2),
    ("sst2", "glue", "sst2", 2),
]

def tokenize(example, task):
    if task == "cola":
        return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)
    if task == "mrpc":
        return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=128)
    if task == "sst2":
        return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

def load_task(task_tuple):
    name, group, subset, num_labels = task_tuple
    ds = load_dataset(group, subset)
    ds = ds.map(lambda x: tokenize(x, name), batched=True)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids","attention_mask","labels"])
    return ds, num_labels



In [4]:
def build_shared_lora_model(num_labels):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="SEQ_CLS",
    )

    model = get_peft_model(base_model, lora_config)
    return model


In [5]:
def train_task(model, dataset, task_name, num_labels, prev_lora_state=None, reg_lambda=0.0):

    if not hasattr(model, "task_heads"):
        model.task_heads = {}

    if task_name not in model.task_heads:
        model.task_heads[task_name] = nn.Linear(model.config.hidden_size, num_labels)

    model.classifier = model.task_heads[task_name]

    metric = evaluate.load("glue", task_name)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return metric.compute(predictions=preds, references=labels)

    training_args = TrainingArguments(
        output_dir=f"./{task_name}_output",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        compute_metrics=compute_metrics,
    )

    if reg_lambda > 0 and prev_lora_state is not None:

        original_compute_loss = trainer.compute_loss

        def compute_loss_with_reg(model, inputs, return_outputs=False):
            loss, outputs = original_compute_loss(model, inputs, return_outputs=True)

            reg_loss = 0.0
            for n, p in model.named_parameters():
                if "lora_" in n and n in prev_lora_state:
                    reg_loss += torch.norm(p - prev_lora_state[n])**2

            loss = loss + reg_lambda * reg_loss
            return (loss, outputs) if return_outputs else loss

        trainer.compute_loss = compute_loss_with_reg

    trainer.train()

    current_lora_state = {
        n: p.detach().clone()
        for n, p in model.named_parameters()
        if "lora_" in n
    }

    return trainer, current_lora_state


In [6]:
def evaluate_all_tasks(model, trained_tasks):
    results = {}
    for task_name, ds in trained_tasks.items():
        model.classifier = model.task_heads[task_name]
        metric = evaluate.load("glue", task_name)

        trainer = Trainer(model=model)
        preds = trainer.predict(ds["validation"])
        logits = preds.predictions
        labels = preds.label_ids
        preds = np.argmax(logits, axis=-1)
        score = metric.compute(predictions=preds, references=labels)
        results[task_name] = score
    return results


In [ ]:
# =========================
# Run Continual Experiment
# =========================

model = build_shared_lora_model(num_labels=2)
# The Trainer will handle placing the model on the correct device(s)
# model.to(device) # Removed this line

trained_tasks = {}
prev_lora = None
forgetting_table = {}

for task_name, group, subset, num_labels in tasks:

    ds, num_labels = load_task((task_name, group, subset, num_labels))
    trained_tasks[task_name] = ds

    print(f"\nTraining on {task_name}")

    trainer, prev_lora = train_task(
        model,
        ds,
        task_name,
        num_labels,
        prev_lora_state=prev_lora,
        reg_lambda=0.01   # Shared subspace stability
    )

    results = evaluate_all_tasks(model, trained_tasks)
    forgetting_table[task_name] = results

print("\n=== Forgetting Table ===")
for step, scores in forgetting_table.items():
    print(step, scores)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

...

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]


Training on cola


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
